# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sohamrathi17/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Lane:** Refresh / Content Opportunity Scoring (from ML-02).

**Task type: ranking / scoring, built on top of a classifier.**

The editor's question is "which pages first?", and that is a ranking question. A yes/no answer would not help much on its own: more than half of the pages in my slice are declining (shown below), so a classifier that just says "declining" hands the editor thousands of pages with no order.

So the plan has two parts:
1. **A classifier** estimates, for each page, the probability that it is declining, using only signals that were knowable before the outcome.
2. **A scoring step** turns that into a priority by combining *how likely* the page is to be declining with *how much visibility is at stake*. Then I sort each client's pages by that score.

The output is a ranked review queue per client, not a single global list. Each client's editor works through their own pages.

It is not clustering (I have a clear outcome to aim at) and not pure signal analysis (the goal is an ordered list someone acts on, not a report about signals).

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/sohamrathi17/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import pandas as pd, numpy as np
raw = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The lane's slice: pages with enough previous-window visibility for a trend to mean something.
VOLUME_FLOOR = 100  # impressions in the previous 30 days
lane = raw[raw["impressions_prev_30d"] >= VOLUME_FLOOR].copy()
lane["is_declining"] = lane["trend_direction"].eq("down").astype(int)

print(f"Full starter data: {len(raw):,} pages, {raw['client_id'].nunique()} clients")
print(f"Lane slice (prev-30d impressions >= {VOLUME_FLOOR}): {len(lane):,} pages, {lane['client_id'].nunique()} clients")
print(f"Declining in the slice: {lane['is_declining'].mean():.1%}  <- too common for a yes/no list to set priorities")

Full starter data: 30,000 pages, 32 clients
Lane slice (prev-30d impressions >= 100): 18,010 pages, 30 clients
Declining in the slice: 61.6%  <- too common for a yes/no list to set priorities


## 2. Target or proxy

I need two columns, because the score combines two things:

| Column | What it is | Role |
|---|---|---|
| `is_declining` | 1 if impressions in the last 30 days are more than 20% below the 30 days before (`trend_direction == "down"`), else 0 | The **classification target** |
| `impressions_lost` | `impressions_prev_30d - impressions_last_30d`, floored at 0 | The **impact** of a decline, used to *evaluate* the queue (how much of the real loss it catches) |

**Observed outcome or defined rule?** Mostly observed, with one defined piece. The impression counts are measured outcomes from Search Console, not a score someone invented. The -20% cut-off is a definition, though, so I will check that results do not flip if I move it (e.g. -10% or -30%).

**Why I call this a proxy for now.** In the starter data, the outcome window (the last 60 days) sits *inside* the same 90 days the features describe. A model trained on it describes a trend that is already happening, it does not forecast one. That is fine for learning the mechanics, but the capstone target will come from the warehouse daily table: **features from months M-3 to M-1, outcome measured in month M+1**, with the final month kept sealed as a test month.

**Leakage rules for the target:** `trend_direction`, `trend_pct`, `impressions_last_30d` and `clicks_last_30d` are the answer (or contain it) and are never features. `impressions_lost` is for evaluation only.

In [2]:
# Sketch the two target columns on the real slice
lane["impressions_lost"] = (lane["impressions_prev_30d"] - lane["impressions_last_30d"]).clip(lower=0)

print("is_declining value counts:")
print(lane["is_declining"].value_counts().rename({1: "declining (1)", 0: "not declining (0)"}).to_string())
print("\nimpressions_lost among declining pages (heavy-tailed, so the median and the max are far apart):")
print(lane.loc[lane["is_declining"] == 1, "impressions_lost"].describe(percentiles=[0.5, 0.9, 0.99]).round(0).to_string())

# Sensitivity of the defined cut-off: how many pages count as declining at other thresholds?
change = (lane["impressions_last_30d"] - lane["impressions_prev_30d"]) / lane["impressions_prev_30d"] * 100
for cut in (-10, -20, -30):
    print(f"Declining at a {cut}% cut-off: {(change < cut).mean():.1%}")

is_declining value counts:
is_declining
declining (1)        11090
not declining (0)     6920

impressions_lost among declining pages (heavy-tailed, so the median and the max are far apart):
count    11090.0
mean      1262.0
std       3348.0
min         22.0
50%        368.0
90%       2813.0
99%      13716.0
max      97995.0
Declining at a -10% cut-off: 70.9%
Declining at a -20% cut-off: 61.6%
Declining at a -30% cut-off: 51.8%


## 3. Success metric

**Main metric: precision@20 per client, averaged across clients I held out from training.**

Of the 20 pages at the top of a client's queue, what share is actually declining? I picked 20 because it is roughly a week of review work for one editor, and the metric is computed per client because each editor only sees their own queue. Clients are held out as a group (client-holdout split) so that the model is tested on sites it has never seen.

**Guard-rail metric: share of lost impressions captured in the top 20.** Precision alone would happily fill the queue with tiny pages that are technically declining. This second number checks that the queue also catches the declines that matter.

**What number means "good".** "Good" means beating the simple rules on *both* numbers at once, on held-out clients. The baselines below already show the bar:
- The base rate is about 61%: random ordering already hits that precision, so any queue below it is worse than doing nothing clever.
- "Biggest pages first" catches the most loss (~46%) but its precision (~0.53) is *below* the base rate.
- "Youngest pages first" sits at about base-rate precision (~0.64) and catches only ~17% of the loss.
- The stale x visible rule from ML-01 scores *exactly* like random order: in this slice almost no page is 180+ days stale, so its top 20 is all ties broken at random.

My target: **precision@20 clearly above the base rate (aim for 0.70+) while capturing at least as much lost visibility as "biggest pages first".** I am fixing this now, before training anything, so I cannot move the goalposts later.

In [3]:
# Baselines on the real slice: macro precision@20 and loss captured@20, per client (clients with >= 20 pages)
K = 20
rng = np.random.default_rng(42)
lane["rand_tiebreak"] = rng.random(len(lane))  # so tied scores are broken randomly, not by file order

baselines = {
    "random order":            lane["rand_tiebreak"],
    "biggest pages first":     lane["impressions_prev_30d"],
    "youngest pages first":    -lane["content_age_days"],
    "stale x visible (ML-01)": ((lane["days_since_last_update"] >= 180) & (lane["impressions_90d"] >= 500)) * lane["impressions_90d"],
}

eligible = lane.groupby("client_id").filter(lambda g: len(g) >= K)
rows = []
for name, score in baselines.items():
    precs, caps = [], []
    for _, g in eligible.assign(score=score).groupby("client_id"):
        top = g.sort_values(["score", "rand_tiebreak"], ascending=False).head(K)
        precs.append(top["is_declining"].mean())
        caps.append(top["impressions_lost"].sum() / max(g["impressions_lost"].sum(), 1))
    rows.append({"ranking": name, "precision@20": np.mean(precs), "loss captured@20": np.mean(caps)})

print(f"Clients with >= {K} pages in the slice: {eligible['client_id'].nunique()}")
print(f"Base rate (average of per-client declining rates): {eligible.groupby('client_id')['is_declining'].mean().mean():.3f}\n")
print(pd.DataFrame(rows).round(3).to_string(index=False))

Clients with >= 20 pages in the slice: 22
Base rate (average of per-client declining rates): 0.613

                ranking  precision@20  loss captured@20
           random order         0.627             0.188
    biggest pages first         0.527             0.463
   youngest pages first         0.639             0.174
stale x visible (ML-01)         0.627             0.188


## 4. The unit of analysis, as a real dataframe

**One row = one page (content item) of one client, described over a trailing 90-day window.**

- `content_id` is unique, so there is exactly one row per page (checked below).
- `client_id` is only for grouping: building each client's queue and splitting train/test by client. It is never a feature.
- The features shown are the safe, knowable-before-the-outcome kind: age, update history, 90-day visibility and position, content size, engagement.
- The last two columns are the target and its impact. In the real model they are never fed in as inputs.

In [4]:
assert lane["content_id"].is_unique, "more than one row per page, the grain is wrong"
print(f"Grain check passed: {len(lane):,} rows = {lane['content_id'].nunique():,} unique pages\n")

id_cols      = ["content_id", "client_id"]
feature_cols = ["content_age_days", "days_since_last_update", "impressions_90d", "avg_position",
                "ctr", "word_count", "content_type", "engagement_rate"]
target_cols  = ["is_declining", "impressions_lost"]

unit = lane[id_cols + feature_cols + target_cols]
print("Pages per client in the slice:", unit.groupby("client_id").size().describe()[["min", "50%", "max"]].to_dict())
unit.sample(8, random_state=42)

Grain check passed: 18,010 rows = 18,010 unique pages

Pages per client in the slice: {'min': 1.0, '50%': 150.0, 'max': 5690.0}


,content_id,client_id,content_age_days,days_since_last_update,impressions_90d,avg_position,ctr,word_count,content_type,engagement_rate,is_declining,impressions_lost
7014,content_bdccfa73c698,client_6208ef0f77,125,104,12059,34.5,0.12,3825.0,keyword article,0.57,1,1169
16642,content_2d12a2ae6fc7,client_f74efabef1,112,20,10428,5.3,0.24,2359.0,keyword article,10.26,1,5139
8230,content_1dff5ee30a0c,client_6208ef0f77,216,104,6729,24.2,0.15,6571.0,keyword article,0.00,1,1431
21905,content_14f02f6b2395,client_6208ef0f77,223,104,404,3.9,0.00,4785.0,keyword article,0.00,1,112
17635,content_feb5c7b065f6,client_2c624232cd,438,22,656,70.6,0.00,NaN,keyword article,0.00,0,0
1202,content_c4178658755d,client_19581e27de,482,22,1880,7.9,1.38,NaN,keyword article,3.40,1,254
11373,content_880ca7127bdf,client_f74efabef1,112,20,982,5.3,0.31,2668.0,keyword article,0.00,1,89
15788,content_bcaa614da3ad,client_434c9b5ae5,113,8,627,8.5,0.16,2525.0,keyword article,0.00,1,500


## 5. Why ML beats a fixed rule here

Three reasons, each backed by the numbers in this notebook:

1. **Each simple rule only gets one half of the job right** (section 3). "Biggest pages first" finds where the loss is but is wrong more often than random. "Youngest" is right about as often as random but misses most of the loss, and "stale" barely fires at all. The ranking I need has to weigh both at once, and finding the right trade-off from data is what a model does.
2. **The signals interact, so a single if-statement points the wrong way for some pages.** In the table below, pages older than a year decline only 24% of the time when they rank deep, but 65% of the time in the top 3. Age matters, but how much depends on position. A fixed rule like "flag pages older than X" cannot capture that; a tree-based model can, and I can still read it (as in notebook 02).
3. **It has to work on clients it has never seen.** In ML-01 a hand rule that looked great in-sample lost most of its edge on held-out clients, while the trees held up. Patterns shift between sites, so the method has to be fit and validated on the data, not hand-tuned once.

**What ML does *not* get to skip.** ML only earns its place if it beats the baselines above on held-out clients (that is ML-07 and ML-08). And the output stays decision-support: it tells an editor where to look, then the editor chooses the action.

**Tying it to the content action.** Each row in the final queue gets a reason code and a suggested action:
- **Refresh:** visible and losing impressions on page 1.
- **Expand:** losing ground in striking distance (positions 11-20).
- **Protect:** high-visibility page holding steady, so leave it alone.
- **Monitor:** small or noisy decline.

In [5]:
# Evidence for reason 2: the decline rate depends on age AND position together (observed, this slice only)
age_order = ["31-90", "91-180", "181-365", "365+"]
pos_order = ["top_3", "page_1", "striking", "page_3_5", "deep"]
rate  = pd.crosstab(lane["age_tier"], lane["position_tier"], values=lane["is_declining"], aggfunc="mean")
count = pd.crosstab(lane["age_tier"], lane["position_tier"])
print("Declining rate by age tier x position tier:")
print(rate.reindex(index=age_order, columns=pos_order).round(2).to_string())
print("\nPages per cell (small cells are not reliable, e.g. the 31-90 day row):")
print(count.reindex(index=age_order, columns=pos_order).to_string())

Declining rate by age tier x position tier:
position_tier  top_3  page_1  striking  page_3_5  deep
age_tier                                              
31-90           1.00    0.72      0.79      0.68  0.50
91-180          0.82    0.65      0.74      0.79  0.60
181-365         0.71    0.59      0.63      0.63  0.44
365+            0.65    0.51      0.48      0.36  0.24

Pages per cell (small cells are not reliable, e.g. the 31-90 day row):
position_tier  top_3  page_1  striking  page_3_5  deep
age_tier                                              
31-90              5     138        39        41     4
91-180           228    3223      1968      1644   104
181-365          185    2329      1726      2047   117
365+              40    1709      1163      1057   243


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled: markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere (pseudonymized IDs and aggregate numbers only)
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/`, then submit your repo URL on the card. Done.